In [1]:
using Statistics, GDAL, DataFrames, CSV, Parquet, GeoParquet

In [2]:
DATA_PREFIX = "../data"
OUTPUT_PREFIX = "../output"

TRAIN_CSV = "$DATA_PREFIX/train.csv"
TRAIN_PARQUET = "$OUTPUT_PREFIX/train.parquet"
TEST_CSV = "$DATA_PREFIX/test.csv"
SUBMISSION_CSV = "$DATA_PREFIX/sample_submission.csv"

"../data/sample_submission.csv"

In [13]:
train_df = CSV.read(TRAIN_CSV, DataFrames.DataFrame)
train_df

Row,Column1,x,y,b02_s2.cdse.quarterly_xxxx0401_xxxx0631,b04_s2.cdse.quarterly_xxxx0401_xxxx0631,b08_s2.cdse.quarterly_xxxx0401_xxxx0631,obs_s2.cdse.quarterly_xxxx0401_xxxx0631,HH_dB_palsar_xxxx,HV_dB_palsar_xxxx,angle_palsar_xxxx,qa_palsar_xxxx,vv.sentinel1_xxxx07,vh.sentinel1_xxxx07,biomass,bioregion,forest_type,year,tile_id
,Int64,Float64,Float64,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64,Float64,Float64,Int64,Int64
1,0,3.13553e6,1.81107e6,297.0,388.0,2642.0,11.0,-5.10893,-9.99938,52.7048,255.0,1493.0,453.0,-9999.0,9.0,10.0,2020,38052
2,1,3.13712e6,1.8108e6,306.0,422.0,2632.0,10.0,-7.45217,-12.6394,51.0869,255.0,1097.0,502.0,28.7421,9.0,9.0,2020,38052
3,2,3.13619e6,1.81023e6,339.0,506.0,2366.0,9.0,-7.14926,-11.7571,48.2711,255.0,1690.0,394.0,-9999.0,9.0,10.0,2020,38052
4,3,3.13808e6,1.80854e6,292.0,380.0,3400.0,8.0,-7.56663,-12.5438,47.8257,255.0,2066.0,579.0,61.379,9.0,9.0,2020,38052
5,4,3.13683e6,1.8099e6,453.0,606.0,2684.0,11.0,-7.49188,-11.671,43.8081,255.0,2264.0,418.0,41.4457,9.0,9.0,2020,38052
6,5,3.13601e6,1.80872e6,425.0,589.0,2394.0,9.0,-8.95152,-12.7943,48.1546,255.0,1982.0,571.0,-9999.0,9.0,9.0,2020,38052
7,6,3.13647e6,1.81084e6,481.0,622.0,2222.0,10.0,-5.77613,-12.3169,58.4605,255.0,2874.0,608.0,-9999.0,9.0,9.0,2020,38052
8,7,3.13692e6,1.80845e6,878.0,1184.0,2354.0,10.0,-6.35796,-13.4901,44.0722,255.0,3023.0,797.0,-9999.0,9.0,0.0,2020,38052
9,8,3.13759e6,1.81082e6,570.0,795.0,2452.0,9.0,-7.99502,-14.8693,26.7398,255.0,1638.0,407.0,25.9117,9.0,9.0,2020,38052


In [14]:
dropmissing!(train_df)
train_df

Row,Column1,x,y,b02_s2.cdse.quarterly_xxxx0401_xxxx0631,b04_s2.cdse.quarterly_xxxx0401_xxxx0631,b08_s2.cdse.quarterly_xxxx0401_xxxx0631,obs_s2.cdse.quarterly_xxxx0401_xxxx0631,HH_dB_palsar_xxxx,HV_dB_palsar_xxxx,angle_palsar_xxxx,qa_palsar_xxxx,vv.sentinel1_xxxx07,vh.sentinel1_xxxx07,biomass,bioregion,forest_type,year,tile_id
,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64,Int64
1,0,3.13553e6,1.81107e6,297.0,388.0,2642.0,11.0,-5.10893,-9.99938,52.7048,255.0,1493.0,453.0,-9999.0,9.0,10.0,2020,38052
2,1,3.13712e6,1.8108e6,306.0,422.0,2632.0,10.0,-7.45217,-12.6394,51.0869,255.0,1097.0,502.0,28.7421,9.0,9.0,2020,38052
3,2,3.13619e6,1.81023e6,339.0,506.0,2366.0,9.0,-7.14926,-11.7571,48.2711,255.0,1690.0,394.0,-9999.0,9.0,10.0,2020,38052
4,3,3.13808e6,1.80854e6,292.0,380.0,3400.0,8.0,-7.56663,-12.5438,47.8257,255.0,2066.0,579.0,61.379,9.0,9.0,2020,38052
5,4,3.13683e6,1.8099e6,453.0,606.0,2684.0,11.0,-7.49188,-11.671,43.8081,255.0,2264.0,418.0,41.4457,9.0,9.0,2020,38052
6,5,3.13601e6,1.80872e6,425.0,589.0,2394.0,9.0,-8.95152,-12.7943,48.1546,255.0,1982.0,571.0,-9999.0,9.0,9.0,2020,38052
7,6,3.13647e6,1.81084e6,481.0,622.0,2222.0,10.0,-5.77613,-12.3169,58.4605,255.0,2874.0,608.0,-9999.0,9.0,9.0,2020,38052
8,7,3.13692e6,1.80845e6,878.0,1184.0,2354.0,10.0,-6.35796,-13.4901,44.0722,255.0,3023.0,797.0,-9999.0,9.0,0.0,2020,38052
9,8,3.13759e6,1.81082e6,570.0,795.0,2452.0,9.0,-7.99502,-14.8693,26.7398,255.0,1638.0,407.0,25.9117,9.0,9.0,2020,38052


In [15]:
PREDICTORS = [
    # "row_id",
    # "x",
    # "y",
    "b02_s2.cdse.quarterly_xxxx0401_xxxx0631",
    "b04_s2.cdse.quarterly_xxxx0401_xxxx0631",
    "b08_s2.cdse.quarterly_xxxx0401_xxxx0631",
    "obs_s2.cdse.quarterly_xxxx0401_xxxx0631",
    "HH_dB_palsar_xxxx",
    "HV_dB_palsar_xxxx",
    "angle_palsar_xxxx",
    # "qa_palsar_xxxx",
    "vv.sentinel1_xxxx07",
    "vh.sentinel1_xxxx07",
    "bioregion",
    "forest_type",
    # "year",
    # "tile_id",
]

LABEL = "biomass"

"biomass"

In [18]:
for col in PREDICTORS
    train_df[!, col] = Float32.(train_df[!, col])
end

train_df

Row,Column1,x,y,b02_s2.cdse.quarterly_xxxx0401_xxxx0631,b04_s2.cdse.quarterly_xxxx0401_xxxx0631,b08_s2.cdse.quarterly_xxxx0401_xxxx0631,obs_s2.cdse.quarterly_xxxx0401_xxxx0631,HH_dB_palsar_xxxx,HV_dB_palsar_xxxx,angle_palsar_xxxx,qa_palsar_xxxx,vv.sentinel1_xxxx07,vh.sentinel1_xxxx07,biomass,bioregion,forest_type,year,tile_id
,Int64,Float64,Float64,Float32,Float32,Float32,Float32,Float32,Float32,Float32,Float64,Float32,Float32,Float64,Float32,Float32,Int64,Int64
1,0,3.13553e6,1.81107e6,297.0,388.0,2642.0,11.0,-5.10893,-9.99938,52.7048,255.0,1493.0,453.0,-9999.0,9.0,10.0,2020,38052
2,1,3.13712e6,1.8108e6,306.0,422.0,2632.0,10.0,-7.45217,-12.6394,51.0869,255.0,1097.0,502.0,28.7421,9.0,9.0,2020,38052
3,2,3.13619e6,1.81023e6,339.0,506.0,2366.0,9.0,-7.14926,-11.7571,48.2711,255.0,1690.0,394.0,-9999.0,9.0,10.0,2020,38052
4,3,3.13808e6,1.80854e6,292.0,380.0,3400.0,8.0,-7.56663,-12.5438,47.8257,255.0,2066.0,579.0,61.379,9.0,9.0,2020,38052
5,4,3.13683e6,1.8099e6,453.0,606.0,2684.0,11.0,-7.49188,-11.671,43.8081,255.0,2264.0,418.0,41.4457,9.0,9.0,2020,38052
6,5,3.13601e6,1.80872e6,425.0,589.0,2394.0,9.0,-8.95152,-12.7943,48.1546,255.0,1982.0,571.0,-9999.0,9.0,9.0,2020,38052
7,6,3.13647e6,1.81084e6,481.0,622.0,2222.0,10.0,-5.77613,-12.3169,58.4605,255.0,2874.0,608.0,-9999.0,9.0,9.0,2020,38052
8,7,3.13692e6,1.80845e6,878.0,1184.0,2354.0,10.0,-6.35796,-13.4901,44.0722,255.0,3023.0,797.0,-9999.0,9.0,0.0,2020,38052
9,8,3.13759e6,1.81082e6,570.0,795.0,2452.0,9.0,-7.99502,-14.8693,26.7398,255.0,1638.0,407.0,25.9117,9.0,9.0,2020,38052


In [20]:
# generate NDVI
NEW_PREDICTORS = [PREDICTORS..., "NDWI", "NDVI", "RVI_PALSAR", "RVI_S1"]

transform!(
  train_df, ["b08_s2.cdse.quarterly_xxxx0401_xxxx0631", "b04_s2.cdse.quarterly_xxxx0401_xxxx0631"] => ((b1, b2) -> (b1 - b2) / (b1 + b2)) => :NDVI)

train_df
# train_df["NDVI"] = (
#     train_df["b08_s2.cdse.quarterly_xxxx0401_xxxx0631"]
#     - train_df["b04_s2.cdse.quarterly_xxxx0401_xxxx0631"]
# ) * (
#     train_df["b08_s2.cdse.quarterly_xxxx0401_xxxx0631"]
#     + train_df["b04_s2.cdse.quarterly_xxxx0401_xxxx0631"]
# ) * 1e4

# train_df["NDWI"] = (
#     (
#         train_df["b02_s2.cdse.quarterly_xxxx0401_xxxx0631"]
#         - train_df["b08_s2.cdse.quarterly_xxxx0401_xxxx0631"]
#     )
#     * (
#         train_df["b02_s2.cdse.quarterly_xxxx0401_xxxx0631"]
#         + train_df["b08_s2.cdse.quarterly_xxxx0401_xxxx0631"]
#     )
#     * 1e4
# )

# train_df["RVI_PALSAR"] = (
#     (train_df["HH_dB_palsar_xxxx"] - train_df["HV_dB_palsar_xxxx"])
#     * (train_df["HH_dB_palsar_xxxx"] + train_df["HV_dB_palsar_xxxx"])
#     * 1e4
# )

# train_df["RVI_S1"] = (
#     (train_df["vv.sentinel1_xxxx07"] - train_df["vh.sentinel1_xxxx07"])
#     * (train_df["vv.sentinel1_xxxx07"] + train_df["vh.sentinel1_xxxx07"])
#     * 1e4
# )

OutOfMemoryError: OutOfMemoryError()